# Designing effective, honest visuals

_Data Wrangling_

**Strip a chart down to the data, encode it so the eye reads it accurately, and the reader gets the message in seconds.**

A chart is a visual sentence: it should say one thing, and say it fast. Edward Tufte gave
       the guiding ratio &mdash; the data-ink ratio. Picture all the ink (pixels) in a chart and
       split it into ink that shows the data and ink that does not (gridlines, borders, 3-D walls,
       background fills). Tufte's rule: maximize the data-ink, erase the rest. Every non-data mark
       is "chart junk" the reader's eye has to wade through.

       The second idea is the encoding hierarchy. Cleveland and McGill measured how accurately
       people read quantities from different visual channels. The ranking, best first, is roughly:
       position > length > angle > area > color shade. So a value drawn as a bar's
       length is read far more accurately than the same value drawn as a pie slice's angle
       or a bubble's area. This single ranking is why "use a bar chart, not a pie chart" is the most
       repeated advice in visualization.

---

This notebook rebuilds a cluttered chart into a clean one, one step at a time, on real bundled data. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — matplotlib + seaborn

We draw the same seven numbers twice — first the cluttered matplotlib default, then a
decluttered version — so the two sit side by side. We build it in three steps: (1) the
data and the figure, (2) the BEFORE chart, (3) the AFTER chart.

### Step 1 — Pull the data and set up the side-by-side figure

Take the mean of seven chemical features across the class-0 wines — a small pandas
Series mapping each feature name to a value. We create one figure with two axes so the
cluttered "before" and the clean "after" can be compared directly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine

# --- real, bundled data: mean chemistry of class-0 wines ---
wine = load_wine(as_frame=True)
df = wine.frame
feats = ['alcohol', 'malic_acid', 'ash', 'total_phenols',
         'flavanoids', 'color_intensity', 'hue']
means = df[df.target == 0][feats].mean()   # a pandas Series, label -> value

fig, (ax_bad, ax_good) = plt.subplots(1, 2, figsize=(13, 5))

### Step 2 — Draw the cluttered "before" chart

This is the matplotlib default taken to its worst: bars in dataframe order (no
ranking), a different rainbow color per bar (color carries no signal), a full grid and
every spine (chart junk), a vague title with no takeaway, and cramped rotated labels.
Notice how hard it is to read the ranking off this.

In [ ]:
# ============ BEFORE: the cluttered default ============
# Unsorted (dataframe order), a different rainbow color per bar,
# full grid, every spine, a vague title, no value labels.
rainbow = plt.cm.rainbow(np.linspace(0, 1, len(means)))
ax_bad.bar(means.index, means.values, color=rainbow)
ax_bad.grid(True)                          # chart junk: gridlines
ax_bad.set_title('Wine features')          # vague, no takeaway
plt.setp(ax_bad.get_xticklabels(), rotation=90)   # cramped labels

### Step 3 — Draw the decluttered "after" chart

Now fix every problem at once. Sort the bars by value so rank is instant; draw them
horizontally with one muted base color and a single color-blind-safe accent on the top
bar; write the title as the conclusion; label each bar's value directly so no legend or
grid is needed; and strip the spines and ticks — maximizing the data-ink ratio.

In [ ]:
# ============ AFTER: decluttered, sorted, labeled ============
s = means.sort_values(ascending=False)     # sort bars by value

# one muted base color, a single color-blind-safe accent on the top bar
base = '#bdbdbd'
accent = sns.color_palette('colorblind')[1]        # safe orange
colors = [accent if i == 0 else base for i in range(len(s))]

bars = ax_good.barh(s.index[::-1], s.values[::-1], color=colors[::-1])
ax_good.set_xlabel('mean value')           # units / axis label
ax_good.set_title('Alcohol dominates the average chemistry of class-0 wines',
                  loc='left')              # title states the conclusion

# direct value labels at the end of each bar -> no legend, no grid needed
for bar, v in zip(bars, s.values[::-1]):
    ax_good.text(v + 0.1, bar.get_y() + bar.get_height() / 2,
                 f'{v:.2f}', va='center')

# remove chart junk: drop the top/right (and here all) spines and the grid
sns.despine(ax=ax_good, left=True, bottom=True)
ax_good.tick_params(left=False, bottom=False)
ax_good.set_xticks([])

plt.tight_layout()
plt.show()

## Visualize the data & results

_Take seven real numbers (the mean chemistry of class-0 wines) and draw them the cluttered default way vs the cleaned way. Does the cleaned chart let you read the ranking faster?_

### Step 1 — Compute the seven means in dataframe order

Load the wines, keep the 59 class-0 rows, and average the same seven features. Printed
in dataframe order, the values arrive in no particular ranking — exactly what the
cluttered chart showed.

In [ ]:
import numpy as np
from sklearn.datasets import load_wine

# 178 real wines, 13 chemical features; take the 59 class-0 wines
wine = load_wine(as_frame=True)
df = wine.frame
feats = ['alcohol', 'malic_acid', 'ash', 'total_phenols',
         'flavanoids', 'color_intensity', 'hue']

means = df[df.target == 0][feats].mean()
print('default (dataframe) order:')
print(means.round(2).to_dict())
# -> {'alcohol': 13.74, 'malic_acid': 2.01, 'ash': 2.46, 'total_phenols': 2.84,
#     'flavanoids': 2.98, 'color_intensity': 5.53, 'hue': 1.06}

### Step 2 — Sort by value to reveal the ranking

Sorting the same Series descending is the one change that lets the eye read the order
instantly — alcohol, then color_intensity, then flavanoids, and so on. This is the
ordering the cleaned chart draws.

In [ ]:
print('sorted by value (what the cleaned chart shows):')
print(means.sort_values(ascending=False).round(2).to_dict())
# -> {'alcohol': 13.74, 'color_intensity': 5.53, 'flavanoids': 2.98,
#     'total_phenols': 2.84, 'ash': 2.46, 'malic_acid': 2.01, 'hue': 1.06}

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate shows market share for 8 product lines as a 3-D pie chart with a rainbow palette and a side legend. Two slices look almost the same size and you can't tell which is bigger. List the design problems and the single best fix.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Name the encoding problem: a pie encodes value as angle/area, the channels the eye reads worst, and 3-D distorts the slices further. — _Cleveland&ndash;McGill show angle and area give large, biased reading errors; two near-equal slices become impossible to rank._
- Name the color problem: a rainbow has no order and likely isn't color-blind-safe; the legend forces the eye to bounce. — _Color carries no real signal here, and red&ndash;green pairs vanish for color-blind readers._
- Replace it with a single sorted bar chart, direct-labeled with the share values, one accent color, no 3-D. — _Length on a common axis is read accurately, sorting makes rank instant, and direct labels remove the legend hunt._

**Answer:** Problems: value is encoded by angle/area (a pie), worsened by 3-D, with a rainbow, non-color-blind-safe palette and a legend the eye must jump to. The single best fix is a sorted, direct-labeled bar chart &mdash; length on a shared axis is the most accurate encoding, sorting reveals rank instantly, and printing the values kills the legend. Drop the 3-D and the rainbow.

</details>

**Problem 2.** You must color a choropleth-style bar chart by a value that ranges from $-40\%$ to $+60\%$ change. A colleague suggests a rainbow scale. Which palette family is correct and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify that the quantity is centered on a meaningful zero (no change), with negatives and positives. — _Whether a value is above or below zero is the message, so zero should be the visual midpoint._
- Reject the rainbow: it has no perceptual order and isn't color-blind-safe. — _A rainbow implies arbitrary jumps and breaks for color-blind readers; it can't show "centered on zero"._
- Choose a diverging palette (e.g. blue&ndash;white&ndash;orange) anchored at zero. — _Two hues meeting at a neutral midpoint map naturally to negative vs positive, and orange&ndash;blue is color-blind-safe._

**Answer:** Use a diverging palette anchored at zero (for example blue for negative, neutral at $0$, orange for positive) &mdash; the quantity is centered on a meaningful midpoint, so the colors should be too. A sequential ramp would be right for a $0$-to-max quantity, and a categorical palette for unordered groups; a rainbow is wrong on both order and accessibility.

</details>

**Problem 3.** Your bar chart distinguishes "treatment" from "control" only by a red bar and a green bar, with no labels. Why is this risky, and what two small changes fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Spot that color is the only channel separating the two groups. — _If color fails, the reader cannot tell the groups apart at all._
- Note that red&ndash;green is the most common color-blindness pairing, and printouts may be grayscale. — _Roughly 1 in 12 men can't reliably separate red from green; the chart would be meaningless to them._
- Add a redundant cue &mdash; direct text labels on each bar &mdash; and switch to a color-blind-safe pair. — _Now the distinction survives without color, and the colors are readable for everyone._

**Answer:** It relies on color alone, using the red&ndash;green pair that is invisible to the most common color blindness (and to a grayscale printout). Two fixes: (1) add a redundant non-color cue &mdash; direct text labels (or different hatching/markers) on each bar &mdash; so the groups are distinguishable without color, and (2) swap red&ndash;green for a color-blind-safe pair such as orange and blue.

</details>